# DEA Wetlands Insight Tool: multi-scene processing with STAC

The Wetlands Insight Tool (WIT) summarises changes in open water, wetness, green vegetation, dry vegetation and bare soil. This notebook follows the [DEA WIT notebook](https://github.com/GeoscienceAustralia/dea-notebooks/blob/stable/DEA_products/DEA_Wetlands_Insight_Tool.ipynb), but processes large wetlands spanning multiple Landsat path/rows using the time aggregation approach from [wit_tooling](https://github.com/GeoscienceAustralia/wit_tooling). No Datacube index is required: observations are discovered using DEA's STAC API and read from public DEA storage.

Each feature in the input shapefile is analysed separately. The output is a normalised WIT graph and CSV for each feature. The calculation uses Landsat Collection 3 products; it should not be expected to reproduce an older Collection 2 WIT product exactly.

## Background

Wetlands support biodiversity, water quality and carbon storage. WIT provides a consistent history of surface water and vegetation changes to help investigate wetland responses to floods, drought and management. Open water, wetness and dry land are mutually exclusive at each valid pixel; green vegetation, dry vegetation and bare soil describe fractions of the dry-land component. Interpret changes alongside field knowledge and the quality information in the CSV.

### Expected output

A stacked plot with open water, wetness, green vegetation, dry vegetation and bare soil, in the same style as the [published WIT graph](https://dea-public-data.s3-ap-southeast-2.amazonaws.com/derivative/ga_ls_wit_ramsar_class_myear_3/1-0-0/1986--P35Y/271.png). Vegetation is normalised for display so the five proportions sum to one wherever the vegetation total is nonzero. A CSV retains both original and normalised proportions, the missing-pixel fraction and the number of scenes in each composite.

## WIT calculation and multi-scene aggregation

For each polygon, Landsat surface reflectance, fractional cover (FC) and Water Observations from Space (WOfS) items are matched by satellite, path/row and acquisition time. In the spirit of `wit_tooling`'s `--aggregate 15` mode, the first available scene starts a 15-day window. Scenes with acquisition times strictly before the end of that window fill pixels that were not valid in earlier scenes; the first subsequent scene starts the next window. The composite is summarised only after scenes have been combined, so adjacent path/rows and observations on different days can contribute to one result.

WOfS identifies open water and excludes cloud, cloud shadow and non-contiguous pixels. On non-water pixels, Landsat fmask and valid reflectance are required; Tasseled Cap Wetness (TCW) at or above −350 is wet, and other pixels contribute fractional cover. Earlier valid open water or land is not overwritten by later observations. Like WIT tooling, the polygon raster includes every touched pixel and a plotted composite must have **less than 10% missing**. Unlike the original code's per-band aggregation, the first valid WIT class is kept together at each pixel so that a later dry observation cannot also be counted over earlier water.

To bound memory, the polygon is rasterised and processed in tiles; only the per-window summaries are retained. Completed windows are checkpointed and reused on rerun. A large polygon and a multi-year range can require extensive network transfer and hours or longer; start with a short range first.

## Getting Started

### Load packages

Use a kernel with this repository installed, for example `pip install -e '.[notebook]'` or `uv pip install --python /path/to/venv/bin/python -e '.[notebook]'`. The notebook requires network access to DEA STAC and public S3 data. The helper module implements the scene matching, tiled loading and aggregation shown below; its source is [`dea_conflux/wit_stac.py`](../dea_conflux/wit_stac.py).

In [ ]:
from pathlib import Path
import logging

import geopandas as gpd
import odc.stac
import pandas as pd
import pystac_client
from IPython.display import display
from odc.geo.geom import Geometry

from dea_conflux.wit_stac import search_scenes, run_windows, normalise_results, plot_results

logging.basicConfig(level=logging.INFO)

### Connect to DEA's STAC API

Configure unsigned access to public S3 assets, then open the DEA Explorer STAC catalogue. Unlike the Datacube-index based Conflux command, this notebook does not need a database connection.

In [ ]:
odc.stac.configure_s3_access(cloud_defaults=True, aws_unsigned=True)
catalog = pystac_client.Client.open("https://explorer.dea.ga.gov.au/stac")

### Analysis parameters

Set `shapefile` to a polygon or multipolygon dataset, `id_field` to a unique, non-null feature identifier, and choose a date range. The start is inclusive and the end is **exclusive**: the default covers the two years from 1 January 2022 up to 1 January 2024. The example uses `fid` from the supplied Ramsar shapefile. Set `id_field = None` to use the file's row index instead, or set `name_field` to a column to use as the plot title.

For a historical analysis, try `start_date = "1986-01-01"` and an end date after the latest desired day. Availability depends on the catalogue and satellite record. The Landsat 5 and 7 inclusion periods follow the historical predicates in `wit_tooling` (including Landsat 7 SLC-off observations during 2010–2013); Landsat 8 and 9 are included when available. They are not identical to the single-scene notebook's Landsat 7 filter. Test a shorter range before attempting the full history.

In [ ]:
shapefile = Path.home() / "ramsarbig/ramsarbig.shp"
id_field = "fid"
name_field = None
start_date = "2022-01-01"
end_date = "2024-01-01"  # exclusive
window_days = 15
resolution = 30  # metres, EPSG:3577
tile_size = 1024  # pixels; decrease if memory is limited
output_dir = Path("wit_multiscene_output")

### Load a shapefile that describes your wetland study area

Use 2D polygon geometries with no Z coordinates, as in the original notebook. Geometries are reprojected to Australian Albers (EPSG:3577). Unlike the original notebook's `poly.geometry[0]`, every feature is processed independently, including overlapping polygons. Identifiers must be unique and safe to use as filenames.

In [ ]:
poly = gpd.read_file(shapefile)
if poly.crs is None:
    raise ValueError("The shapefile must have a coordinate reference system")
if poly.empty or not poly.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Provide at least one polygon or multipolygon")
if poly.geometry.is_empty.any() or not poly.geometry.is_valid.all():
    raise ValueError("All polygon geometries must be non-empty and valid")
if poly.geometry.has_z.any():
    raise ValueError("Remove Z coordinates before processing")
poly = poly.to_crs("EPSG:3577")
if id_field is not None:
    poly = poly.set_index(id_field)
if poly.index.hasnans or not poly.index.is_unique or len(set(map(str, poly.index))) != len(poly):
    raise ValueError("Feature IDs must be unique and non-null")
for feature_id in poly.index:
    name = str(feature_id)
    if name in ("", ".", "..") or "/" in name or "\\" in name:
        raise ValueError(f"Feature ID cannot be used as a filename: {name!r}")
if name_field is not None and name_field not in poly.columns:
    raise ValueError(f"Missing name field: {name_field}")
output_dir.mkdir(parents=True, exist_ok=True)
display(poly)

## Load Landsat, Water Observations and Fractional Cover

### Search for matching observations

Each polygon is searched independently using its WGS84 bounds and footprint. The STAC search covers Landsat 5, 7, 8 and 9 ARD, WOfS and FC. An ARD item without both matching FC and WOfS is logged and skipped; conflicting duplicate items cause an error rather than mixing scenes. Search results are sorted by acquisition time before making windows.

In [ ]:
scene_groups = {}
for feature_id, feature in poly.iterrows():
    gpgon = Geometry(feature.geometry, str(poly.crs))
    scene_groups[feature_id] = search_scenes(catalog, gpgon, start_date, end_date)
    print(f"{feature_id}: {len(scene_groups[feature_id])} matched Landsat/FC/WOfS scenes")
    if not scene_groups[feature_id]:
        raise ValueError(f"No complete observations for polygon {feature_id}")

### Load and mask data; calculate Tasseled Cap Wetness

For each tile and scene, the helper loads WOfS using nearest-neighbour resampling, reflectance using bilinear resampling, and fmask/FC using nearest-neighbour resampling. It computes the Crist (1985) TCW index from the six NBART bands, applies the −350 threshold and masks cloud, shadow, non-contiguous and nodata pixels. FC values are divided by 100. Open-water pixels need valid WOfS; dry-land and wet pixels additionally need valid reflectance and fmask.

### Composite and summarise

`run_windows` builds first-valid composites in chronological order, sums each WIT component across the rasterised polygon, and divides by the number of valid pixels. `pc_missing` is relative to all polygon pixels. This is equivalent to processing the common full grid but avoids loading the entire grid in memory. The checkpoint key includes the geometry, resolution, algorithm version and input scene IDs. Re-running the same analysis reuses completed windows, including after an interruption; changing the input data or geometry recomputes affected windows.

In [ ]:
polygon_results = {}
for feature_id, feature in poly.iterrows():
    print(f"Processing polygon {feature_id}")
    polygon_results[feature_id] = run_windows(
        scene_groups[feature_id], feature.geometry,
        cache_dir=output_dir / "checkpoints",
        crs=str(poly.crs), resolution=resolution,
        tile_size=tile_size, window_days=window_days,
    )

## Normalise Fractional Cover Values in WIT result

As in the original notebook, normalisation does not change open water or wetness. It rescales the three FC values to the remaining vegetation area, `1 − water − wet`. Both unnormalised and normalised values are retained. If the FC total is zero while vegetation area remains, normalisation is undefined and the normalised values are reported as missing rather than inventing a composition.

In [ ]:
polygon_results = {
    feature_id: normalise_results(frame)
    for feature_id, frame in polygon_results.items()
}
for feature_id, frame in polygon_results.items():
    print(f"Polygon {feature_id}")
    display(frame.head())

## Create WIT Plot

Each plot is displayed and saved as `<feature_id>.png` in `output_dir`. As in WIT tooling, observations with 10% or more missing pixels are excluded from the graph; they remain in the CSV for quality assessment. If no observation passes this threshold, the saved graph is labelled accordingly. The single-observation case is shown over a 24-hour span to make the stacked areas visible. Shorter-than-one-year plots use monthly labels; longer plots use yearly labels.

In [ ]:
for feature_id, frame in polygon_results.items():
    polygon_name = str(poly.loc[feature_id, name_field]) if name_field else f"Wetland {feature_id}"
    plot_results(frame, polygon_name, output_dir / str(feature_id))

## Create WIT Comma-separated values (CSV) output file

The CSV contains one row per composite window. Dates are the UTC acquisition times of the first scene in each window; `scene_count` is the number of matched scenes in that window. Fractions use the range 0–1 where applicable. `pc_missing` includes clouds, unobserved pixels and pixels not covered by a valid scene. `norm_pv`, `norm_npv` and `norm_bs` are the normalised green, dry and bare-soil components.

In [ ]:
for feature_id, frame in polygon_results.items():
    csv_path = output_dir / f"{feature_id}.csv"
    frame.to_csv(csv_path, index=False)
    print(f"Wrote {csv_path} and {output_dir / (str(feature_id) + '.png')}")

## Additional information

- [WIT tooling and aggregation](https://github.com/GeoscienceAustralia/wit_tooling)
- [DEA Wetlands Insight Tool information](https://knowledge.dea.ga.gov.au/data/product/dea-wetlands-insight-tool/)
- Dunn et al. (2023), [Wetlands Insight Tool](https://doi.org/10.1007/s13157-023-01682-7)
- Crist (1985), [Tasseled Cap transformation](https://doi.org/10.1016/0034-4257(85)90102-6)

Notebook code is licensed under Apache 2.0. DEA data are licensed under Creative Commons Attribution 4.0.